In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np


# Define CNN Feature Extractor

def build_cnn_model(input_shape=(28, 28, 1)):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu', name="feature_dense")(x)
    outputs = layers.Dense(10, activation='softmax')(x)
    model = models.Model(inputs, outputs)
    return model

cnn_model = build_cnn_model()
print("CNN model built successfully.")


# Load Dataset (Example: MNIST)

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = np.expand_dims(x_train.astype("float32") / 255.0, -1)
x_test = np.expand_dims(x_test.astype("float32") / 255.0, -1)

cnn_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# quick training for demonstration
cnn_model.fit(x_train[:10000], y_train[:10000], epochs=1, batch_size=128, validation_split=0.1, verbose=1)


# Create feature extractor

feature_extractor = models.Model(
    inputs=cnn_model.input,
    outputs=cnn_model.get_layer("feature_dense").output
)
print("Feature extractor created successfully!")

# Extract features
train_features = feature_extractor.predict(x_train[:5000], batch_size=256)
test_features = feature_extractor.predict(x_test[:1000], batch_size=256)

print(f"Features extracted: Train={train_features.shape}, Test={test_features.shape}")


# Define Simple KAN Layer and Model

class KANLayer(tf.keras.layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.units = units

    def build(self, input_shape):
        self.W = self.add_weight(shape=(input_shape[-1], self.units),
                                 initializer="glorot_uniform", trainable=True)
        self.b = self.add_weight(shape=(self.units,), initializer="zeros", trainable=True)
        self.alpha = self.add_weight(shape=(self.units,), initializer="ones", trainable=True)

    def call(self, inputs):
        linear = tf.matmul(inputs, self.W) + self.b
        nonlinear = tf.math.sin(self.alpha * linear)
        return linear + nonlinear

def build_kan_model(input_dim, num_classes):
    inputs = layers.Input(shape=(input_dim,))
    x = KANLayer(64)(inputs)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs, outputs)

kan_model = build_kan_model(train_features.shape[1], 10)
kan_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
kan_model.summary()


# Train KAN on extracted CNN features

kan_model.fit(train_features, y_train[:5000], epochs=5, batch_size=128, validation_split=0.1)

# Evaluate
test_loss, test_acc = kan_model.evaluate(test_features, y_test[:1000])
print(f"\nKAN Test Accuracy: {test_acc:.4f}")


✅ CNN model built successfully.
71/71 ━━━━━━━━━━━━━━━━━━━━ 4s 42ms/step - accuracy: 0.6704 - loss: 1.1548 - val_accuracy: 0.9300 - val_loss: 0.2400
✅ Feature extractor created successfully!
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
Features extracted: Train=(5000, 128), Test=(1000, 128)


Model: "functional_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_10 (InputLayer)     │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ kan_layer (KANLayer)            │ (None, 64)             │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,130 (51.29 KB)

 Trainable params: 13,130 (51.29 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.4960 - loss: 1.7069 - val_accuracy: 0.9100 - val_loss: 0.3472
Epoch 2/5
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9281 - loss: 0.2900 - val_accuracy: 0.9260 - val_loss: 0.2473
Epoch 3/5
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9420 - loss: 0.2108 - val_accuracy: 0.9380 - val_loss: 0.2197
Epoch 4/5
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9444 - loss: 0.1888 - val_accuracy: 0.9380 - val_loss: 0.2146
Epoch 5/5
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9476 - loss: 0.1736 - val_accuracy: 0.9420 - val_loss: 0.2053
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9352 - loss: 0.1796 

✅ KAN Test Accuracy: 0.9240


In [ ]:
import os


# Save Trained Models


os.makedirs("models", exist_ok=True)

# Save CNN feature extractor (Keras .h5 format)
cnn_model.save("models/cnn_model.h5")
feature_extractor.save("models/feature_extractor.h5")

# Save KAN model (Keras .h5 format)
kan_model.save("models/kan_model.h5")

print("\nModels saved successfully in the 'models/' folder!")
print("Files created:")
print(" - models/cnn_model.h5")
print(" - models/feature_extractor.h5")
print(" - models/kan_model.h5")



💾 Models saved successfully in the 'models/' folder!
📦 Files created:
 - models/cnn_model.h5
 - models/feature_extractor.h5
 - models/kan_model.h5
